In [2]:
# q1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
n = 200

df = pd.DataFrame({
    "bedrooms": np.random.randint(1, 6, n),
    "bathrooms": np.random.randint(1, 4, n),
    "sqft": np.random.randint(500, 3500, n),
    "age": np.random.randint(0, 50, n),
    "neighborhood": np.random.choice(["A", "B", "C"], n)
})

df["price"] = (
    df["sqft"] * 150 +
    df["bedrooms"] * 10000 +
    df["bathrooms"] * 8000 -
    df["age"] * 1000 +
    np.random.randint(-20000, 20000, n)
)

# CLEANING

df = df.dropna()

# ENCODING
df = pd.get_dummies(df, columns=["neighborhood"], drop_first=True)

# FEATURES / TARGET
y = df["price"]
x = df.drop(columns=["price"])

# TRAIN TEST SPLIT
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

# SCALING 
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# LINEAR REGRESSION
lr_model = LinearRegression()
lr_model.fit(x_train_scaled, y_train)
lr_pred = lr_model.predict(x_test_scaled)

lr_r2 = r2_score(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

print("\nLINEAR REGRESSION")
print("R2:", lr_r2)
print("RMSE:", lr_rmse)

# DECISION TREE
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(x_train, y_train)
dt_pred = dt_model.predict(x_test)

dt_r2 = r2_score(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

print("\nDECISION TREE")
print("R2:", dt_r2)
print("RMSE:", dt_rmse)

# FEATURE IMPORTANCE (coefficients)
importance = pd.DataFrame({"Feature": x.columns, "Coefficient": lr_model.coef_
}).sort_values(by="Coefficient", ascending=False)

print("\nFEATURE IMPORTANCE")
print(importance)

# PREDICT NEW HOUSE
new_house = pd.DataFrame([{"bedrooms": 3, "bathrooms": 2, "sqft": 1500, "age": 10, "neighborhood_B": 0, "neighborhood_C": 0}])

new_house_scaled = scaler.transform(new_house)

pred_price = lr_model.predict(new_house_scaled)

print("\nNEW HOUSE PRICE PREDICTION:", pred_price[0])


LINEAR REGRESSION
R2: 0.9886077764927323
RMSE: 13387.99831178475

DECISION TREE
R2: 0.9422896403981759
RMSE: 30132.71685359499

FEATURE IMPORTANCE
          Feature    Coefficient
2            sqft  124165.688915
0        bedrooms   16510.454541
1       bathrooms    6107.059371
5  neighborhood_C    -257.343469
4  neighborhood_B   -2678.492942
3             age  -14487.267500

NEW HOUSE PRICE PREDICTION: 262476.0205158444


In [4]:
#q2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# CREATE SAMPLE DATASET
data = {
    "email_text": [
        "Congratulations you won a free lottery click here",
        "Meeting scheduled for tomorrow at office",
        "Claim your free prize now!!!",
        "Please find the project report attached",
        "Earn money fast working from home click link",
        "Let's have lunch tomorrow",
        "You have been selected for a cash reward",
        "Can we reschedule our meeting?",
        "Win a brand new car now",
        "Here is the invoice for your purchase"
    ],
    "label": [1,0,1,0,1,0,1,0,1,0]
}

df = pd.DataFrame(data)

# PREPROCESSING (TEXT → NUMERIC)
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df["email_text"])

y = df["label"]

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# MODEL TRAINING
model = LogisticRegression()
model.fit(X_train, y_train)

# PREDICTION & EVALUATION
y_pred = model.predict(X_test)

print("\nACCURACY:", accuracy_score(y_test, y_pred))

print("\nCLASSIFICATION REPORT:\n", classification_report(y_test, y_pred, zero_division=0))

# DEPLOYMENT (NEW EMAIL PREDICTION)
new_emails = [
    "You won a free iPhone click now",
    "Please send me the meeting notes"
]

new_X = vectorizer.transform(new_emails)
predictions = model.predict(new_X)

print("\nNEW EMAIL PREDICTIONS:")
for email, pred in zip(new_emails, predictions):
    print(f"{email} -> {'SPAM' if pred == 1 else 'NOT SPAM'}")


ACCURACY: 0.3333333333333333

CLASSIFICATION REPORT:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3


NEW EMAIL PREDICTIONS:
You won a free iPhone click now -> SPAM
Please send me the meeting notes -> NOT SPAM


In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
n = 200

df = pd.DataFrame({
    "total_spending": np.random.randint(500, 20000, n),
    "age": np.random.randint(18, 70, n),
    "visits": np.random.randint(1, 50, n),
    "frequency": np.random.randint(1, 20, n)
})

# Target: high value customer (1) or low value (0)
df["label"] = (
    (df["total_spending"] > 10000) &
    (df["visits"] > 20) &
    (df["frequency"] > 8)
).astype(int)

# CLEANING (simulate missing + fix)
df.loc[np.random.choice(df.index, 5), "age"] = np.nan
df["age"] = df["age"].fillna(df["age"].median())

# OUTLIER HANDLING 
for col in ["total_spending", "visits", "frequency"]:
    df[col] = np.clip(df[col], df[col].quantile(0.05), df[col].quantile(0.95))

# FEATURES & TARGET
X = df.drop("label", axis=1)
y = df["label"]

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

# FEATURE SCALING 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LINEAR SVM
model = SVC(kernel="linear")
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

# EVALUATION
print("\nACCURACY:", accuracy_score(y_test, y_pred))

print("\nCLASSIFICATION REPORT:\n", classification_report(y_test, y_pred))

weights = model.coef_[0]
intercept = model.intercept_[0]

features = X.columns

print("\nHYPERPLANE EQUATION:")
equation = " + ".join([f"{w:.2f}*{f}" for w, f in zip(weights, features)])
print(f"{equation} + ({intercept:.2f}) = 0")

print("\nRULES FOR CLASSIFICATION:")
print("- High value customer if:")
print("  • High spending")
print("  • AND high visits")
print("  • AND high purchase frequency")

print("\nExample rule extracted from model:")
print("IF spending > threshold AND visits > threshold THEN High Value (1)")


ACCURACY: 0.925

CLASSIFICATION REPORT:
               precision    recall  f1-score   support

           0       0.94      0.97      0.96        35
           1       0.75      0.60      0.67         5

    accuracy                           0.93        40
   macro avg       0.85      0.79      0.81        40
weighted avg       0.92      0.93      0.92        40


HYPERPLANE EQUATION:
1.21*total_spending + -0.16*age + 1.05*visits + 0.82*frequency + (-1.91) = 0

RULES FOR CLASSIFICATION:
- High value customer if:
  • High spending
  • AND high visits
  • AND high purchase frequency

Example rule extracted from model:
IF spending > threshold AND visits > threshold THEN High Value (1)
